# 面向推理服务的KVCache
KVCache 和显存，预设的请求batch_size, seq_len, num_kv_heads, head_dim, layer_size 有关
所以初始化的时候一般会申请2个（K和V）5维tensor
1. num_layers: 注意力层数
2. batch_size: 处理批次量
3. seq_len 上下文长度
4. num_kv_heads: 注意力头数
5. head_dim 头维度

request_to_slot 用于查询KV-CACHE
slot_to_request: 查询请求的现有token_id序列
free_slots: 可用坑位slot_id集合，用于判断是否可以加入新的处理请求

In [ ]:
import torch
class KVCacheManager:
    # 当前的kvcache 的问题是： 固定了max_batch_size 和 max_seq_len， 所以有可能造成浪费。因为请求长度不一，有长有短，短请求较多就浪费了。
    def __init__(self, config):
        self.k_cache = torch.zeros(config.num_layers, config.max_batch_size, config.max_seq_len, config.num_kv_heads, config.head_dim)
        self.v_cache = torch.zeros(config.num_layers, config.max_batch_size, config.max_seq_len, config.num_kv_heads, config.head_dim)
        # 记录每个请求推理到的len，这个是动态变化的，因为随着decode，这个长度不断的在加
        self.sequence_lengths = torch.zeros(config.max_batch_size, dtype=torch.long)
        self.request_to_slot = {} #request id --> slot_index, 当前没有PagedKVCache，所以是一对一的
        self.slot_to_request = {} # slot_index -> request id
        self.free_slots = set(range(config.max_batch_size)) # kvcache 所有的槽位，因为有max_batch_size请求上线，所以这里也是max_batch_size个
    

    def update_slots(self, slot_ids, new_kv_cache):
        print(f"update slots: {slot_ids}")
        # 按照层更新kvcache
        for i, layer_kv_cache in enumerate(new_kv_cache):
            # 获取这一批请求的batcj_size，对齐的sequence长度，kv头, 头维度的大小
            batch_size, seq_len, num_kv_heads, head_dim = layer_kv_cache[0].shape
            if seq_len == 1: # 代表是Decoding
                # 每个序列request的当前的长度，decode会不断的加， slots_ids 一一对应requests
                cur_len = self.sequence_lengths[slot_ids]
                self.k_cache[i, slot_ids, cur_len, :, :] = layer_kv_cache[0][:,0,:,:]
                self.v_cache[i, slot_ids, cur_len, :, :] = layer_kv_cache[1][:,0,:,:]
            else: # prefill
                self.k_cache[i, slot_ids, :seq_len, :, :] = layer_kv_cache[0]
                self.v_cache[i, slot_ids, :seq_len, :, :] = layer_kv_cache[1]
        

    def free_slot(self, request_id: int):
        print(f"free request_id {request_id}, free it's kvcache")
        if request_id in self.request_to_slot:
            slot_id = self.request_to_slot[request_id]
            del self.request_to_slot[request_id]
            del self.slot_to_request[slot_id]

            # 释放槽位，更新k_cahce, v_cache
            self.free_slots.add(slot_id)
            self.k_cache[:, slot_id, :, :, :] = 0
            self.v_cache[:, slot_id, :, :, :] = 0


In [7]:
from random import randint
import torch

a = torch.range(start=10, end=100, step=5)
print(a)

b =  [1,2,3]
print(a[b])
print('-----')
c = randint(2,10)
b = torch.randint(10,(1, c))
print(b)

tensor([ 10.,  15.,  20.,  25.,  30.,  35.,  40.,  45.,  50.,  55.,  60.,  65.,
         70.,  75.,  80.,  85.,  90.,  95., 100.])
tensor([15., 20., 25.])
-----
tensor([[8, 6, 7, 8, 0, 0, 5]])


/tmp/ipykernel_2982970/1107706040.py:4: UserWarning: torch.range is deprecated and will be removed in a future release because its behavior is inconsistent with Python's range builtin. Instead, use torch.arange, which produces values in [start, end).
  a = torch.range(start=10, end=100, step=5)


Engine 主要作用：
1. 接收到别人抛来的请求，添加到对列（request manager 统一管理）
2. 提供step方法执行prefill或者decode

In [ ]:
from typing import List

# 核心是step方法
class ContinueBatchingEngine:

    def __init__(self, model, config) -> None:
        self.kv_cache_manager = KVCacheManager(config)
        self.model = ModelWrapper(model, self.kv_cache_manager)
        self.request_manager = RequestManager(config.max_batch_size,)

    def add_request(self, prompt: List[int], max_seq_len: int) -> int:
        return self.request_manager.add_request(prompt, max_seq_len)

    # 优先decode, 其次prefill
    # 这里体现continue batching， 先decoding, 在prefill
    # 这里假设每次step 增加一个请求
    # step1 prefill req1
    # step2 decoding req1 prefill req2
    # step3 decoding req1,req2, prefill req3
    def step(self):
        # decoding阶段(已经有请求)
        if has_activate_requests():
            # 获取kvcache的槽和运行中的requests_ids
            activate_slots, request_ids = get_activate_slots_info()
            # 获取最新token
            input_tokens = get_input_ids(self.request_manager, request_ids)
            # 进行decode, 同时更新kvcache也在decode的程序内部
            decoding_logits = self.model.decode(input_tokens, activate_slots,)

            next_tokens = generate_next_tokens(decoding_logits)

            # 关联request_id 和 token，然后更新状态
            for i, request_id in enumerate(request_ids):
                # 更新状态
                update_request(request_id, next_tokens[i])
                update_slots(request_id) # 如果decode到eos 或者 达到max length，则结束释放slot

        # prefill 新请求
        if has_available_slots():
            pending_requests = get_pending_requests()
            if pending_requests:
                prefill_logits = self.model.prefill(pending_requests)
                prefill_tokens = generate_next_tokens(prefill_logits)
                # 更新状态
                for i, (request_id, _) in enumerate(pending_requests):
                    update_request(request_id, prefill_tokens[i].item())

主进程主要是不断循环执行step(prefill or decoce)
另一个就是不断循环看有没有请求，把请求送给engine

for true request ---> engine ---> for true step

In [ ]:



#  模拟伪代码接口

model = ToyModel(config)
engine = ContinueBatchingEngine(model, config)

while True:
    engine.step()

# listen
engine_pid = get_inference_engine_pid()
while True:
    req = get_new_request()
    send(engine_pid, 'add_request', req)